# Master Benchmark Suite: Full 10-Model Evaluation for Drug `N05C`

Collects holdout predictions across all candidate models for drug `N05C` on the 2019 Test set:
* **Part A: Point Forecast Accuracy Benchmark Table (Sorted by RMSLE)**
* **Part B: Enterprise Probabilistic Demand Range Deliverable ($[P_{10}, P_{50}, P_{90}]$)**


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'N05C'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for N05C loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: Collect All Holdout Model Predictions & Display Benchmark Ladder
m0 = pd.read_csv('m0_naive_preds.csv')['pred_Naive'].values
m1 = pd.read_csv('m1_arima_preds.csv')['pred_ARIMA'].values
m2 = pd.read_csv('m2_ets_preds.csv')['pred_ETS'].values
m3a = pd.read_csv('m3_sarima_preds.csv')['pred_SARIMA'].values
m3b = pd.read_csv('m3_sarimax_preds.csv')['pred_SARIMAX'].values
m4 = pd.read_csv('m4_prophet_preds.csv')['pred_Prophet'].values
m5 = pd.read_csv('m5_lstm_preds.csv')['pred_LSTM'].values
m6 = pd.read_csv('m6_lightgbm_preds.csv')['pred_LightGBM'].values
m7 = pd.read_csv('m7_xgb_quantile_preds.csv')['pred_XGB_Quantile'].values
m8 = pd.read_csv('m8_tft_preds.csv')['pred_TFT'].values

model_dict = {
    'Model 6: LightGBM + SHAP (Optuna)': m6,
    'Model 7: XGBoost Quantile (Optuna)': m7,
    'Model 4: Meta Prophet': m4,
    'Model 2: Holt-Winters ETS': m2,
    'Model 8: TFT / Deep Attention': m8,
    'Model 5: PyTorch LSTM': m5,
    'Model 1: Classical ARIMA': m1,
    'Model 3b: SARIMAX + Exog': m3b,
    'Model 3a: Pure SARIMA': m3a,
    'Model 0: Optimised Naive (k*=365)': m0
}

records = []
for name, preds in model_dict.items():
    met = evaluate_metrics(test_series.values, preds)
    met['Model'] = name
    records.append(met)

benchmark_df = pd.DataFrame(records)[['Model', 'RMSLE', 'RMSE', 'MAE', 'WAPE (%)']].sort_values('RMSLE').reset_index(drop=True)
benchmark_df.index = benchmark_df.index + 1

print("==========================================================================")
print(f"  PART A: POINT FORECAST BENCHMARK LADDER ({TARGET_DRUG} — 2019 HOLDOUT TEST SET)")
print("==========================================================================")
display(benchmark_df)

champion = benchmark_df.iloc[0]
print("\nChampion Model for Point Forecasting:")
print(f"  * #1 Rank Model : {champion['Model']}")
print(f"  * RMSLE         : {champion['RMSLE']:.6f}")
print(f"  * RMSE          : {champion['RMSE']:.4f}")
print(f"  * MAE           : {champion['MAE']:.4f}")
print(f"  * WAPE (%)      : {champion['WAPE (%)']:.2f}%")


  PART A: POINT FORECAST BENCHMARK LADDER (N05C — 2019 HOLDOUT TEST SET)


,Model,RMSLE,RMSE,MAE,WAPE (%)
1,Model 1: Classical ARIMA,0.519835,1.119028,0.813356,116.608689
2,Model 2: Holt-Winters ETS,0.520796,1.116638,0.814664,116.796238
3,Model 3a: Pure SARIMA,0.521129,1.111267,0.820481,117.630244
4,Model 3b: SARIMAX + Exog,0.524373,1.137197,0.800487,114.763725
5,Model 5: PyTorch LSTM,0.524593,1.106843,0.832114,119.297942
6,Model 4: Meta Prophet,0.525873,1.132085,0.806425,115.614977
7,Model 6: LightGBM + SHAP (Optuna),0.534999,1.171388,0.773072,110.833254
8,Model 8: TFT / Deep Attention,0.539757,1.108019,0.856963,122.860536
9,Model 7: XGBoost Quantile (Optuna),0.561706,1.206769,0.752530,107.888202
10,Model 0: Optimised Naive (k*=365),0.722157,1.544131,1.024911,146.938776



Champion Model for Point Forecasting:
  * #1 Rank Model : Model 1: Classical ARIMA
  * RMSLE         : 0.519835
  * RMSE          : 1.1190
  * MAE           : 0.8134
  * WAPE (%)      : 116.61%


In [3]:
# Step 2: Part B — Enterprise Probabilistic Demand Range Deliverable
hybrid_plan = pd.read_csv('n05c_hybrid_supply_chain_plan.csv')

print("==========================================================================")
print(f"  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE ({TARGET_DRUG})")
print("==========================================================================")
print("First 10 Days Actionable Pack Order Ranges:")
display(hybrid_plan[['Date', 'Actual Sales', 'Lean Lower Bound (P10)', 'Expected Demand Anchor (P50)', 'Upper Target Stock (P90)', 'Order Range (Lean P10 Pack Target)', 'Order Range (Expected P50 Pack Target)', 'Order Range (Safety P90 Pack Target)']].head(10))

service_level = np.mean(test_series.values <= hybrid_plan['Upper Target Stock (P90)'].values) * 100
print(f"\nDeliverable Performance Metrics:")
print(f"  * Achieved P90 Inventory Service Level: {service_level:.2f}% (Target >= 95%)")
print(f"  * Average Daily Uncertainty Range    : {hybrid_plan['Uncertainty Band Width (P90 - P10)'].mean():.2f} units/day")


  PART B: ENTERPRISE PROBABILISTIC DEMAND RANGE DELIVERABLE (N05C)
First 10 Days Actionable Pack Order Ranges:


,Date,Actual Sales,Lean Lower Bound (P10),Expected Demand Anchor (P50),Upper Target Stock (P90),Order Range (Lean P10 Pack Target),Order Range (Expected P50 Pack Target),Order Range (Safety P90 Pack Target)
0,2019-01-01,0.0,0.03,0.33,6.15,1,1,7
1,2019-01-02,0.0,0.05,0.36,5.84,1,1,6
2,2019-01-03,0.0,0.01,0.40,5.93,1,1,6
3,2019-01-04,0.0,0.00,0.40,5.94,0,1,6
4,2019-01-05,0.0,0.00,0.30,5.12,0,1,6
5,2019-01-06,2.0,0.00,0.28,5.37,0,1,6
6,2019-01-07,0.0,0.03,0.45,4.17,1,1,5
7,2019-01-08,0.0,0.06,0.39,4.20,1,1,5
8,2019-01-09,3.0,0.06,0.42,4.28,1,1,5
9,2019-01-10,2.0,0.00,0.43,4.98,0,1,5



Deliverable Performance Metrics:
  * Achieved P90 Inventory Service Level: 98.93% (Target >= 95%)
  * Average Daily Uncertainty Range    : 4.61 units/day
